# Empirical Analysis of ChiCTR Gene Therapy and Gene Editing Clinical Trials

**Associated manuscript**: *Beyond Written Rules: Institutional Failures and Reform Priorities in China's Ethics Governance of Pediatric Gene Editing* 

**Objective**: Using ChiCTR registration data, assess changes in transparency and safety oversight of gene therapy/gene-editing clinical trials before and after the 2023 ethics-review reform.

**Analytical basis**: Section 2.3, Empirical Analysis (Methods), of the manuscript outline.

---

## Analysis workflow

| Step | Description |
|---|---|
| 1. Data loading | Read `ChiCTR.xlsx` |
| 2. Eligibility screening | Apply the Section 2.3 inclusion/exclusion criteria record by record |
| 3. Data cleaning | Parse dates, handle missing data, and encode variables |
| 4. Derived measures | Calculate the approval-to-registration interval, prospective-registration rate, etc. |
| 5. IIT/IND classification | Three-level rule: explicit markers → combined rules → manual review |
| 6. Descriptive statistics | Tables 2a and 2b |
| 7. Visualization | Figure 2: distribution of approval-to-registration intervals |
| 8. Statistical testing | Mann–Whitney U test / Fisher's exact test |
| 9. Sensitivity analyses | Stratify by funding source, study phase, and IIT/IND pathway |
| 10. Results export | Save CSV files and figures |


## 1. Environment setup and package imports


In [ ]:
# === Core data processing ===
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# === Statistical tests ===
from scipy import stats
from scipy.stats import mannwhitneyu, fisher_exact
import statsmodels.stats.proportion as smprop

# === Visualization ===
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# === System utilities ===
import os
import sys

# === Font settings (Windows environment) ===
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False  # Ensure minus signs render correctly

# === Display settings ===
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

print('Package imports complete.')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

## 2. Data loading


In [ ]:
# === File path (update for your local environment) ===
DATA_PATH = r'E:\repo_obsidian\仓库1\global_health\数据资料\ChiCTR.xlsx'

# === Read data ===
df_raw = pd.read_excel(DATA_PATH, engine='openpyxl')

print(f'Original dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'\nColumn names:')
for i, col in enumerate(df_raw.columns):
    print(f'  [{i}] {col}')
print(f'\nPreview of first 5 rows:')
df_raw.head()

In [ ]:
# === Missing values per field ===
print('Missing value statistics:')
missing = df_raw.isnull().sum()
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
for col in df_raw.columns:
    if missing[col] > 0:
        print(f'  {col}: {missing[col]} ({missing_pct[col]}%)')
if missing.sum() == 0:
    print('  No missing values in any field.')
else:
    print(f'\n  Total: {missing.sum()} missing value(s)')

## 3. Eligibility screening

Based on the search and screening strategy in Section 2.3 of the manuscript:

**Inclusion criteria**:

1. Interventional studies (excluding observational studies and diagnostic experiments).
2. In vivo gene therapy or gene editing, including base editing, CRISPR, and AAV-vector-mediated gene delivery.

**Exclusion criteria**:

1. Ex vivo gene-modified cell therapies, such as CAR-T and TCR-T.
2. Non-therapeutic genetic testing or diagnostic studies.
3. Oncolytic viruses, unless transgene expression is involved.

> **Note:** The source dataset may include records that were retrieved but provisionally marked as excluded. Manually review the screened records to confirm that every included record meets the eligibility criteria.

> **Deduplication rule:** Before eligibility screening, records are deduplicated by non-empty ChiCTR registration ID. The first occurrence is retained and subsequent records with the same ID are marked as excluded. Blank registration IDs are not automatically deduplicated and require manual review.


In [ ]:
# === Create screening flags to trace exclusions at each step ===
# Deduplication rule: use the ChiCTR registration ID as the unique identifier;
# retain the first occurrence of each non-empty registration ID.
df = df_raw.copy()
df['Screening_status'] = 'pending'
df['Exclusion_reason'] = ''

# ==========================================
# Step 0: Deduplicate by ChiCTR registration ID
# ==========================================
# Blank/missing registration IDs are not treated as duplicates and must be reviewed manually.
registration_id = df['注册号'].astype('string').str.strip()
mask_duplicate = registration_id.notna() & registration_id.ne('') & registration_id.duplicated(keep='first')

n_duplicate = mask_duplicate.sum()
df.loc[mask_duplicate, 'Screening_status'] = 'excluded'
df.loc[mask_duplicate, 'Exclusion_reason'] = 'Duplicate ChiCTR registration ID'

print('Step 0 — Deduplicate by ChiCTR registration ID:')
nonempty_registration_id = registration_id.notna() & registration_id.ne('')
print(f'  Unique non-empty registration IDs: {registration_id[nonempty_registration_id].nunique()}')
print(f'  Duplicates excluded: {n_duplicate}')
if n_duplicate > 0:
    print('  Duplicate records excluded (first occurrence retained):')
    for idx in df[mask_duplicate].index:
        print(f'    [{idx}] {registration_id.loc[idx]} — {str(df.loc[idx, "注册题目"])[:80]}')

# ==========================================
# Step 1: Retain only interventional studies
# ==========================================
mask_interventional = df['研究类型'].str.contains('干预性研究', na=False)
still_in = df['Screening_status'] == 'pending'
mask_exclude_interventional = still_in & ~mask_interventional

n_not_interventional = mask_exclude_interventional.sum()
df.loc[mask_exclude_interventional, 'Screening_status'] = 'excluded'
df.loc[mask_exclude_interventional, 'Exclusion_reason'] = (
    'Non-interventional: ' + df.loc[mask_exclude_interventional, '研究类型'].fillna('missing')
)

print('Step 1 — Retain interventional studies only:')
print(f'  Retained for subsequent screening: {(df["Screening_status"] == "pending").sum()}')
print(f'  Excluded at this step: {n_not_interventional}')
if n_not_interventional > 0:
    print('  Excluded records:')
    for idx in df[mask_exclude_interventional].index:
        print(f'    [{idx}] {df.loc[idx, "研究类型"]} — {str(df.loc[idx, "注册题目"])[:80]}')


In [ ]:
# ==========================================
# Step 2: Exclude TCR-T / CAR-T (ex vivo cell therapy)
# ==========================================
# Based on whether the study title contains TCR-T or CAR-T keywords
# (these are ex vivo gene-modified cell therapies, not in vivo gene therapy/editing)

tcr_car_keywords = r'TCR-T|CAR-T|TCRT|CART|T cell receptor|嵌合抗原受体'
mask_tcr_car = df['注册题目'].str.contains(tcr_car_keywords, case=False, na=False)

# Only apply this step to records not yet excluded
still_in = df['Screening_status'] == 'pending'
mask_exclude_tcr = still_in & mask_tcr_car

n_tcr_car = mask_exclude_tcr.sum()
df.loc[mask_exclude_tcr, 'Screening_status'] = 'excluded'
df.loc[mask_exclude_tcr, 'Exclusion_reason'] = 'TCR-T/CAR-T (ex vivo cell therapy)'

print(f'Step 2 — Exclude TCR-T/CAR-T:')
print(f'  Excluded: {n_tcr_car}')
if n_tcr_car > 0:
    print(f'  Excluded records:')
    for idx in df[mask_exclude_tcr].index:
        print(f'    [{idx}] {str(df.loc[idx, "注册题目"])[:100]}')

In [ ]:
# ==========================================
# Step 3: Exclude genetic testing/diagnostic studies
# ==========================================
# Studies labelled "diagnostic experiment" were already excluded in Step 1
# Additional check for interventional studies with diagnostic keywords in the title

diagnosis_keywords = r'诊断(?!研究)|biomarker|screening test|diagnostic test|检测方法'
mask_diag = df['注册题目'].str.contains(diagnosis_keywords, case=False, na=False)

still_in = df['Screening_status'] == 'pending'
mask_exclude_diag = still_in & mask_diag

n_diag = mask_exclude_diag.sum()
df.loc[mask_exclude_diag, 'Screening_status'] = 'excluded'
df.loc[mask_exclude_diag, 'Exclusion_reason'] = 'Genetic testing/diagnostic study (non-therapeutic)'

print(f'Step 3 — Exclude genetic testing/diagnostic studies:')
print(f'  Excluded: {n_diag}')
if n_diag > 0:
    for idx in df[mask_exclude_diag].index:
        print(f'    [{idx}] {str(df.loc[idx, "注册题目"])[:100]}')
else:
    print(f'  No records excluded.')

In [ ]:
# ==========================================
# Step 4: Exclude oncolytic viruses (unless involving transgenic expression)
# ==========================================
oncolytic_keywords = r'oncolytic|溶瘤病毒'
mask_oncolytic = df['注册题目'].str.contains(oncolytic_keywords, case=False, na=False)

still_in = df['Screening_status'] == 'pending'
mask_exclude_onco = still_in & mask_oncolytic

n_onco = mask_exclude_onco.sum()
df.loc[mask_exclude_onco, 'Screening_status'] = 'excluded'
df.loc[mask_exclude_onco, 'Exclusion_reason'] = 'Oncolytic virus (no evidence of transgene expression)'

print(f'Step 4 — Exclude oncolytic viruses:')
print(f'  Excluded: {n_onco}')
if n_onco > 0:
    for idx in df[mask_exclude_onco].index:
        print(f'    [{idx}] {str(df.loc[idx, "注册题目"])[:100]}')
else:
    print(f'  No records excluded.')

In [ ]:
# ==========================================
# Screening summary
# ==========================================
df_included = df[df['Screening_status'] == 'pending'].copy()
df_excluded = df[df['Screening_status'] == 'excluded'].copy()

print('=' * 60)
print('SCREENING SUMMARY')
print('=' * 60)
print(f'Original records: {len(df_raw)}')
print(f'Included:         {len(df_included)}')
print(f'Excluded:         {len(df_excluded)}')
print()
print('Exclusion reason distribution:')
for reason, count in df_excluded['Exclusion_reason'].value_counts().items():
    print(f'  {reason}: {count}')

print(f'\n⚠️ Please carefully review the screening results above. If any entries need adjustment, modify them in the manual correction cell below.')

In [ ]:
# ==========================================
# Manual correction area (if needed)
# ==========================================
# Example: to manually exclude a specific record
# manual_exclude_ids = ['ChiCTRxxxxxxxx']
# df_included = df_included[~df_included['注册号'].isin(manual_exclude_ids)]

# Example: to restore a previously excluded record
# manual_include_indices = [17]  # e.g., gene-edited pig kidney transplant
# df_manual = df.loc[manual_include_indices].copy()
# df_included = pd.concat([df_included, df_manual], ignore_index=True)

print(f'After manual correction, final included: {len(df_included)} records')

In [ ]:
# === View included records (for manual audit) ===
print('Included Records:')
print('-' * 80)
for i, (_, row) in enumerate(df_included.iterrows()):
    title = str(row['注册题目'])[:90]
    print(f'{i+1}. [{row["注册号"]}] {title}')
    print(f'    Reg. date: {row["注册时间"]} | Phase: {row["研究所处阶段（是否为IIT代理指标）"]} | DSMB: {row["DSMB存在（有/无)"]}')
    print()

## 4. Data cleaning and variable coding


In [ ]:
# === Reset index ===
df_clean = df_included.reset_index(drop=True).copy()

# ==========================================
# 4.1 Date parsing
# ==========================================
# Registration date → datetime
df_clean['Registration_date_dt'] = pd.to_datetime(df_clean['注册时间'], errors='coerce')

# Ethics committee approval date → datetime
df_clean['Ethics_approval_date_dt'] = pd.to_datetime(df_clean['伦理委员会批准日期'], errors='coerce')

# Check for anomalous dates (e.g., 1990-01-01 is clearly a placeholder)
anomaly_mask = df_clean['Ethics_approval_date_dt'] < pd.Timestamp('2000-01-01')
n_anomaly = anomaly_mask.sum()
if n_anomaly > 0:
    print(f'⚠️ Found {n_anomaly} record(s) with ethics approval date before 2000 — treated as data anomaly/missing:')
    for idx in df_clean[anomaly_mask].index:
        print(f'  [{idx}] {df_clean.loc[idx, "注册号"]}: approval date = {df_clean.loc[idx, "Ethics_approval_date_dt"]}')
    df_clean.loc[anomaly_mask, 'Ethics_approval_date_dt'] = pd.NaT

print(f'\nRegistration date valid: {df_clean["Registration_date_dt"].notna().sum()}/{len(df_clean)}')
print(f'Ethics approval date valid: {df_clean["Ethics_approval_date_dt"].notna().sum()}/{len(df_clean)}')

In [ ]:
# ==========================================
# 4.2 Create grouping variable: pre-reform vs. post-reform
# ==========================================
# Cut-off: February 2023 release of the four-ministry ethics-review measures for human life-science and medical research
# Pre-reform: 2018-03 to 2023-02
# Post-reform: 2023-03 to 2026-07

REFORM_CUTOFF = pd.Timestamp('2023-03-01')

df_clean['Period'] = np.where(
    df_clean['Registration_date_dt'] < REFORM_CUTOFF,
    'Pre-Reform (2018-03 to 2023-02)',
    'Post-Reform (2023-03 to 2026-07)'
)

# If registration date is missing, classify using ethics approval date
missing_reg = df_clean['Registration_date_dt'].isna()
if missing_reg.any():
    df_clean.loc[missing_reg, 'Period'] = np.where(
        df_clean.loc[missing_reg, 'Ethics_approval_date_dt'] < REFORM_CUTOFF,
        'Pre-Reform (2018-03 to 2023-02)',
        'Post-Reform (2023-03 to 2026-07)'
    )

print('Period distribution:')
print(df_clean['Period'].value_counts())
print(f'\nPeriod missing: {df_clean["Period"].isna().sum()}')

In [ ]:
# ==========================================
# 4.3 Derived measure: approval-to-registration interval (days)
# ==========================================
# Registration date − Ethics committee approval date
# Positive = approval before registration (normal workflow)
# Negative = registration before approval (anomalous / retrospective registration)

df_clean['Approval_registration_interval_days'] = (
    df_clean['Registration_date_dt'] - df_clean['Ethics_approval_date_dt']
).dt.days

n_interval = df_clean['Approval_registration_interval_days'].notna().sum()
n_negative = (df_clean['Approval_registration_interval_days'] < 0).sum()

print(f'Records with calculable approval→registration interval: {n_interval}/{len(df_clean)}')
print(f'Of which negative (retrospective registration): {n_negative}')
print(f'\nApproval→Registration interval — descriptive statistics:')
print(df_clean['Approval_registration_interval_days'].describe())

In [ ]:
# ==========================================
# 4.4 Encode categorical variables
# ==========================================

# --- 4.4a Registration status (prospective / retrospective) ---
df_clean['Prospective_registration_binary'] = np.where(
    df_clean['注册状态（预注册/补注册）'].str.contains('预注册', na=False),
    1, 0
)
print(f'Prospective registration: {df_clean["Prospective_registration_binary"].sum()}/{len(df_clean)} ({df_clean["Prospective_registration_binary"].mean()*100:.1f}%)')

# --- 4.4b IPD sharing commitment ---
df_clean['IPD_sharing_binary'] = np.where(
    df_clean['IPD共享承诺（是/否）'].str.strip() == '是',
    1, 0
)
print(f'IPD sharing: {df_clean["IPD_sharing_binary"].sum()}/{len(df_clean)} ({df_clean["IPD_sharing_binary"].mean()*100:.1f}%)')

# --- 4.4c DSMB presence ---
# "有" = 1; "无" or "暂未确定" = 0
df_clean['DSMB_binary'] = np.where(
    df_clean['DSMB存在（有/无)'].str.strip() == '有',
    1, 0
)
print(f'DSMB present: {df_clean["DSMB_binary"].sum()}/{len(df_clean)} ({df_clean["DSMB_binary"].mean()*100:.1f}%)')

# DSMB detailed distribution (may need to report "not yet determined" as a separate category)
print(f'\nDSMB detailed distribution:')
print(df_clean['DSMB存在（有/无)'].value_counts())

In [ ]:
# ==========================================
# 4.4d DSMB "not yet determined" by period (supplementary)
# ==========================================
# DSMB_binary above treats "not yet determined" as absent.
# Here, "not yet determined" is retained as a separate category by period.

dsmb_period = pd.crosstab(
    df_clean['Period'],
    df_clean['DSMB存在（有/无)'].astype(str).str.strip(),
    margins=True
)
print('DSMB category × period crosstab:')
print(dsmb_period)
print()

for period in ['Pre-Reform (2018-03 to 2023-02)', 'Post-Reform (2023-03 to 2026-07)']:
    sub = df_clean[df_clean['Period'] == period]
    n_unknown = (sub['DSMB存在（有/无)'].astype(str).str.strip() == '暂未确定').sum()
    print(f'{period} — DSMB not yet determined: {n_unknown}/{len(sub)} ({n_unknown/len(sub)*100:.1f}%)')


In [ ]:
# ==========================================
# 4.5 Encode funding source
# ==========================================
# Categorize as: Industry / Academic / Mixed / Self-funded

def categorize_funding(fund_str):
    """Classify funding source string into Industry/Academic/Self-funded/Mixed"""
    if pd.isna(fund_str):
        return 'Unknown'
    fund_str = str(fund_str).strip()
    
    # Industry keywords
    corp_keywords = ['公司', '有限', '科技', '生物', '医药', 'Inc', 'Ltd', 'Corp', '企业']
    # Academic/government keywords
    acad_keywords = ['国家自然科学基金', '科委', '科技部', '重点研发', '基金', '项目',
                     '人才', '大学', '医院', '实验室', '创新', '专项']
    # Self-funded keywords
    self_keywords = ['自筹', '自选']
    
    is_corp = any(kw in fund_str for kw in corp_keywords)
    is_acad = any(kw in fund_str for kw in acad_keywords)
    is_self = any(kw in fund_str for kw in self_keywords)
    
    if is_corp and is_acad:
        return 'Mixed (Industry+Academic)'
    elif is_corp:
        return 'Industry'
    elif is_self and not is_acad:
        return 'Self-funded'
    elif is_acad:
        return 'Academic'
    else:
        return 'Self-funded'  # default — please manually verify

df_clean['Funding_category'] = df_clean['经费来源（企业/学术/混合/自筹）'].apply(categorize_funding)

print('Funding source distribution:')
print(df_clean['Funding_category'].value_counts())

# Industry funding binary (for tables)
df_clean['Industry_funded_binary'] = np.where(
    df_clean['Funding_category'].isin(['Industry', 'Mixed (Industry+Academic)']),
    1, 0
)
print(f'\nIndustry-funded (incl. mixed): {df_clean["Industry_funded_binary"].sum()}/{len(df_clean)} ({df_clean["Industry_funded_binary"].mean()*100:.1f}%)')

In [ ]:
# ==========================================
# 4.6 Encode study phase (IIT proxy)
# ==========================================
# Clean study phase labels, removing non-breaking spaces and other invisible characters

df_clean['Study_phase_clean'] = (
    df_clean['研究所处阶段（是否为IIT代理指标）']
    .str.replace('\xa0', '', regex=False)
    .str.strip()
)

# Create IIT proxy flag: "Other" or "Exploratory/Pilot" → suspected IIT
df_clean['Suspected_IIT'] = df_clean['Study_phase_clean'].isin(['其他', '探索性研究/预试验'])

print('Study phase distribution:')
print(df_clean['Study_phase_clean'].value_counts())
print(f'\nSuspected IIT ("Other" + "Exploratory/Pilot"): {df_clean["Suspected_IIT"].sum()}/{len(df_clean)} ({df_clean["Suspected_IIT"].mean()*100:.1f}%)')

## 4.7 IIT/IND classification

The three-level classification rule (see `8_11_IIT分类逻辑.md`) is applied as follows:

- **Level 1:** Explicit markers — IIT/IND keywords in the registered title or funding information.
- **Level 2:** Combined rules — decision matrix based on funding category × study phase.
- **Level 3:** Manual review — return `Uncertain` records to the ChiCTR webpage for verification.


In [ ]:
# ==========================================
# 4.7a Define the IIT/IND classification function
# ==========================================
# Three-level classification rule; see 8_11_IIT分类逻辑.md
# Level 1: Explicit markers in title/funding description
# Level 2: Combinatorial rules (funding category × study phase)
# Level 3: Manual review for Uncertain records

def classify_iit(row):
    """
    Classify a ChiCTR record as IIT, IND, Uncertain, or Exclude.
    
    Parameters
    ----------
    row : pd.Series
        Must contain: 注册题目, 经费来源（企业/学术/混合/自筹）, Funding_category, Study_phase_clean, Industry_funded_binary
    
    Returns
    -------
    str : "IIT", "IND", "Uncertain", or "Exclude"
    """
    title = str(row.get("注册题目", ""))
    funding_desc = str(row.get("经费来源（企业/学术/混合/自筹）", ""))
    funding_cat = str(row.get("Funding_category", ""))
    phase = str(row.get("Study_phase_clean", ""))
    
    # --- Level 1: Explicit markers (case-insensitive) ---
    combined_text = (title + " " + funding_desc).lower()
    
    iit_keywords = [
        "investigator-initiated",
        "investigator initiated",
        "iit",
        "研究者发起",
        "研究者发起的",
        "前沿创新 iit"
    ]
    ind_keywords = ["ind", "新药临床试验", "drug registration"]
    
    has_iit_marker = any(kw in combined_text for kw in iit_keywords)
    has_ind_marker = any(kw in combined_text for kw in ind_keywords)
    
    # Level 1a: explicit IIT marker without conflicting IND marker → IIT
    if has_iit_marker and not has_ind_marker:
        return "IIT"
    # Level 1b: both IIT and IND markers → conflicting signals, manual review
    elif has_iit_marker and has_ind_marker:
        return "Uncertain"
    
    # --- Level 2: Combinatorial rules ---
    # Exclude basic science studies (not clinical trials)
    if phase == "基础科学研究":
        return "Exclude"
    
    # Decision matrix (funding category × study phase)
    if funding_cat == "Industry" and phase in ("I期临床试验", "I期+II期"):
        return "IND"
    if funding_cat == "Industry" and phase in ("探索性研究/预试验", "其他"):
        return "IIT"
    if funding_cat in ("Academic", "Self-funded") and phase in ("探索性研究/预试验", "其他"):
        return "IIT"
    if funding_cat in ("Academic", "Self-funded") and phase in ("I期临床试验", "I期+II期"):
        return "Uncertain"
    if funding_cat == "Mixed (Industry+Academic)":
        return "Uncertain"
    
    # Fallback — should not normally reach here
    return "Uncertain"


# Verify the function is defined
print("✅ classify_iit() function defined.")

In [ ]:
# ==========================================
# 4.7b Apply the IIT/IND classification
# ==========================================

# Apply classification to each record
df_clean['IIT_classification'] = df_clean.apply(classify_iit, axis=1)

# --- Summary ---
print('=' * 60)
print('IIT/IND CLASSIFICATION RESULTS')
print('=' * 60)
print(f'\nOverall distribution (n={len(df_clean)}):')
class_counts = df_clean['IIT_classification'].value_counts()
for cls, cnt in class_counts.items():
    pct = cnt / len(df_clean) * 100
    print(f'  {cls}: {cnt} ({pct:.1f}%)')

# --- Detailed table: each record with its classification ---
print(f'\n{"─" * 90}')
print(f'{"#":<4} {"Registration ID":<22} {"Funding category":<28} {"Study phase":<16} {"Classification":<12} {"Rule triggered"}')
print(f'{"─" * 90}')

for i, (_, row) in enumerate(df_clean.iterrows()):
    reg_id = str(row['注册号']).strip()
    fund_cat = str(row.get('Funding_category', ''))
    phase = str(row.get('Study_phase_clean', ''))
    cls = row['IIT_classification']
    
    # Determine which rule triggered
    title = str(row.get('注册题目', ''))
    fund_desc = str(row.get('经费来源（企业/学术/混合/自筹）', ''))
    combined = (title + ' ' + fund_desc).lower()
    iit_kw = ["investigator-initiated", "investigator initiated", "iit", "研究者发起", "研究者发起的", "前沿创新 iit"]
    has_iit = any(kw in combined for kw in iit_kw)
    
    if has_iit and cls == 'IIT':
        rule = 'Level 1: explicit IIT marker'
    elif cls == 'Exclude':
        rule = 'Level 2: basic science'
    elif cls == 'IND':
        rule = f'Level 2: {fund_cat} + {phase}'
    elif cls == 'IIT':
        rule = f'Level 2: {fund_cat} + {phase}'
    elif cls == 'Uncertain':
        rule = f'Level 2: {fund_cat} + {phase}'
    else:
        rule = '—'
    
    print(f'{i+1:<4} {reg_id:<22} {fund_cat:<28} {phase:<16} {cls:<12} {rule}')

print(f'{"─" * 90}')

# --- Cross-tabulation: IIT_classification × Period ---
print(f'\nCross-tabulation: IIT Classification × Reform Period')
print(f'{"─" * 50}')
ct = pd.crosstab(df_clean['IIT_classification'], df_clean['Period'])
print(ct)
print(f'{"─" * 50}')

## 5. Descriptive statistics: Tables 2a and 2b


In [ ]:
# ==========================================
# 4.7c Transparency indicators stratified by IIT/IND pathway
# ==========================================

print('Transparency and Safety Indicators — Stratified by IIT/IND Pathway')
print('=' * 75)
print('(Excluding "Exclude" and "Uncertain" records)\n')

# Filter to classified records only (IIT or IND)
df_pathway = df_clean[df_clean['IIT_classification'].isin(['IIT', 'IND'])].copy()

for pathway in ['IIT', 'IND']:
    subset = df_pathway[df_pathway['IIT_classification'] == pathway]
    n = len(subset)
    if n == 0:
        print(f'{pathway}: n=0')
        continue
    
    print(f'{pathway} (n={n}):')
    print(f'  Pre-registration: {subset["Prospective_registration_binary"].sum()}/{n} ({subset["Prospective_registration_binary"].mean()*100:.1f}%)')
    print(f'  IPD sharing:      {subset["IPD_sharing_binary"].sum()}/{n} ({subset["IPD_sharing_binary"].mean()*100:.1f}%)')
    print(f'  DSMB present:     {subset["DSMB_binary"].sum()}/{n} ({subset["DSMB_binary"].mean()*100:.1f}%)')
    
    interval = subset['Approval_registration_interval_days'].dropna()
    if len(interval) >= 2:
        print(f'  Interval median:  {interval.median():.0f} days (IQR: {interval.quantile(0.25):.0f}–{interval.quantile(0.75):.0f})')
    print()

# --- Comparison summary table ---
print('─' * 75)
print('Summary Comparison Table')
print('─' * 75)

pathway_summary = []
for pathway in ['IIT', 'IND']:
    subset = df_pathway[df_pathway['IIT_classification'] == pathway]
    n = len(subset)
    if n == 0:
        continue
    pathway_summary.append({
        'Pathway': pathway,
        'n': n,
        'Pre-reg (%)': f'{subset["Prospective_registration_binary"].mean()*100:.1f}',
        'IPD Sharing (%)': f'{subset["IPD_sharing_binary"].mean()*100:.1f}',
        'DSMB (%)': f'{subset["DSMB_binary"].mean()*100:.1f}',
        'Interval Median (d)': f'{subset["Approval_registration_interval_days"].dropna().median():.0f}' if subset['Approval_registration_interval_days'].notna().sum() > 0 else 'N/A',
    })

df_pathway_summary = pd.DataFrame(pathway_summary)
print(df_pathway_summary.to_string(index=False))

# --- Interpretation guidance ---
print(f'\n{"─" * 75}')
print('Interpretation Guidance:')
print('─' * 75)
iit_n = len(df_pathway[df_pathway['IIT_classification'] == 'IIT'])
ind_n = len(df_pathway[df_pathway['IIT_classification'] == 'IND'])
if iit_n > 0 and ind_n > 0:
    iit_ipd = df_pathway[df_pathway['IIT_classification'] == 'IIT']['IPD_sharing_binary'].mean()
    ind_ipd = df_pathway[df_pathway['IIT_classification'] == 'IND']['IPD_sharing_binary'].mean()
    iit_dsmb = df_pathway[df_pathway['IIT_classification'] == 'IIT']['DSMB_binary'].mean()
    ind_dsmb = df_pathway[df_pathway['IIT_classification'] == 'IND']['DSMB_binary'].mean()
    
    print(f'  IIT IPD sharing:  {iit_ipd*100:.1f}%  vs  IND IPD sharing:  {ind_ipd*100:.1f}%')
    print(f'  IIT DSMB present: {iit_dsmb*100:.1f}%  vs  IND DSMB present: {ind_dsmb*100:.1f}%')
    
    if iit_ipd < ind_ipd or iit_dsmb < ind_dsmb:
        print(f'\n  → Potential evidence of "regulatory arbitrage": IIT pathway shows')
        print(f'    lower transparency/safety oversight compared to IND pathway.')
        print(f'    This supports the dual-track arbitrage argument in Discussion 5.3.')
    else:
        print(f'\n  → No evidence of regulatory arbitrage detected at the pathway level.')
        print(f'    Discussion should honestly report: "no pathway-level difference observed,')
        print(f'    suggesting IND transparency may not be systematically better than IIT."')
else:
    print(f'  Insufficient data for pathway comparison (IIT={iit_n}, IND={ind_n}).')

In [ ]:
# ==========================================
# 4.7d Compare the previous IIT proxy with the new IIT/IND classification
# ==========================================
# The old "Suspected_IIT" flag was a simple proxy based solely on study phase
# The new IIT_classification uses the full 3-level rule set
# This cell compares the two to highlight misclassification risk

print('Comparison: Old IIT Proxy vs. New IIT/IND Classification')
print('=' * 65)

# Cross-tabulation
comparison = pd.crosstab(
    df_clean['Suspected_IIT'].map({True: 'Suspected_IIT (old proxy)', False: 'Not IIT proxy'}),
    df_clean['IIT_classification'],
    margins=True
)
print(comparison)
print()

# Identify disagreements
disagree = df_clean[
    (df_clean['Suspected_IIT'] == True) & (df_clean['IIT_classification'] == 'IND')
]
if len(disagree) > 0:
    print(f'⚠️ Records flagged as "suspected IIT" by old proxy but classified as IND ({len(disagree)} records):')
    for _, row in disagree.iterrows():
        print(f'  - {str(row["注册号"]).strip()}: {str(row["注册题目"])[:80]}...')
        print(f'    Funding: {row["Funding_category"]} | Phase: {row["Study_phase_clean"]}')

disagree2 = df_clean[
    (df_clean['Suspected_IIT'] == False) & (df_clean['IIT_classification'] == 'IIT')
]
if len(disagree2) > 0:
    print(f'\n⚠️ Records NOT flagged by old proxy but classified as IIT ({len(disagree2)} records):')
    for _, row in disagree2.iterrows():
        print(f'  - {str(row["注册号"]).strip()}: {str(row["注册题目"])[:80]}...')
        print(f'    Funding: {row["Funding_category"]} | Phase: {row["Study_phase_clean"]}')

print(f'\n→ Old proxy sensitivity: the simple phase-based approach misclassifies')
print(f'  Industry-funded trials in "exploratory/other" phases as IIT, when the')
print(f'  combinatorial rules distinguish them based on funding source.')

## 5.1 Table 2a: Study characteristics by reform period


In [ ]:
# ==========================================
# Table 2a: Baseline characteristics — pre-reform vs. post-reform
# ==========================================

def summarize_period(df_period, label):
    """Calculate summary statistics for a single period"""
    n = len(df_period)
    
    # Study phase
    phase_counts = df_period['Study_phase_clean'].value_counts()
    
    # Basic indicators
    results = {
        'Characteristic': [
            'No. of trials',
            'Phase I (%)',
            'Phase I+II (%)',
            'Exploratory/Pilot (%)',
            'Other (suspected IIT) (%)',
            'Industry-funded (%)',
            'Pre-registration rate (%)',
            'IPD sharing rate (%)',
            'DSMB present (%)',
        ],
        ' ': [
            n,
            f"{phase_counts.get('I期临床试验', 0)} ({phase_counts.get('I期临床试验', 0)/n*100:.1f})" if n > 0 else '0',
            f"{phase_counts.get('I期+II期', 0)} ({phase_counts.get('I期+II期', 0)/n*100:.1f})" if n > 0 else '0',
            f"{phase_counts.get('探索性研究/预试验', 0)} ({phase_counts.get('探索性研究/预试验', 0)/n*100:.1f})" if n > 0 else '0',
            f"{phase_counts.get('其他', 0)} ({phase_counts.get('其他', 0)/n*100:.1f})" if n > 0 else '0',
            f"{df_period['Industry_funded_binary'].sum()} ({df_period['Industry_funded_binary'].mean()*100:.1f})" if n > 0 else '0',
            f"{df_period['Prospective_registration_binary'].sum()} ({df_period['Prospective_registration_binary'].mean()*100:.1f})" if n > 0 else '0',
            f"{df_period['IPD_sharing_binary'].sum()} ({df_period['IPD_sharing_binary'].mean()*100:.1f})" if n > 0 else '0',
            f"{df_period['DSMB_binary'].sum()} ({df_period['DSMB_binary'].mean()*100:.1f})" if n > 0 else '0',
        ]
    }
    return results

# Group calculation
pre_reform = df_clean[df_clean['Period'].str.contains('Pre-Reform', na=False)]
post_reform = df_clean[df_clean['Period'].str.contains('Post-Reform', na=False)]

pre_stats = summarize_period(pre_reform, 'Pre-Reform')
post_stats = summarize_period(post_reform, 'Post-Reform')

# Build combined table
table2a_data = {
    'Characteristic': pre_stats['Characteristic'],
    'Pre-Reform': pre_stats[' '],
    'Post-Reform': post_stats[' '],
}

# Add total column
total_n = len(df_clean)
total_phase = df_clean['Study_phase_clean'].value_counts()
table2a_data['Total'] = [
    total_n,
    f"{total_phase.get('I期临床试验', 0)} ({total_phase.get('I期临床试验', 0)/total_n*100:.1f})",
    f"{total_phase.get('I期+II期', 0)} ({total_phase.get('I期+II期', 0)/total_n*100:.1f})",
    f"{total_phase.get('探索性研究/预试验', 0)} ({total_phase.get('探索性研究/预试验', 0)/total_n*100:.1f})",
    f"{total_phase.get('其他', 0)} ({total_phase.get('其他', 0)/total_n*100:.1f})",
    f"{df_clean['Industry_funded_binary'].sum()} ({df_clean['Industry_funded_binary'].mean()*100:.1f})",
    f"{df_clean['Prospective_registration_binary'].sum()} ({df_clean['Prospective_registration_binary'].mean()*100:.1f})",
    f"{df_clean['IPD_sharing_binary'].sum()} ({df_clean['IPD_sharing_binary'].mean()*100:.1f})",
    f"{df_clean['DSMB_binary'].sum()} ({df_clean['DSMB_binary'].mean()*100:.1f})",
]

table2a = pd.DataFrame(table2a_data)
print('Table 2a: Basic Characteristics of Included Trials')
print('=' * 80)
print(f'Pre-Reform: {len(pre_reform)} trials | Post-Reform: {len(post_reform)} trials | Total: {len(df_clean)} trials')
print('=' * 80)
table2a

In [ ]:
# ==========================================
# Table 2b: Transparency and safety-oversight indicators — with 95% CI
# ==========================================

def proportion_ci(success, n, method='wilson'):
    """Calculate proportion and 95% CI (Wilson method)"""
    if n == 0:
        return np.nan, np.nan, np.nan
    
    if method == 'wilson':
        from statsmodels.stats.proportion import proportion_confint
        ci_low, ci_upp = proportion_confint(success, n, alpha=0.05, method='wilson')
    else:
        from statsmodels.stats.proportion import proportion_confint
        ci_low, ci_upp = proportion_confint(success, n, alpha=0.05, method='beta')
    
    return success/n, ci_low, ci_upp

def format_proportion(numerator, denominator, ci_low, ci_upp):
    """Format as n/N (% [95% CI])"""
    pct = numerator/denominator*100 if denominator > 0 else 0
    return f'{int(numerator)}/{int(denominator)} ({pct:.1f}% [{ci_low*100:.1f}–{ci_upp*100:.1f}])'

indicators = [
    ('Pre-registration rate', 'Prospective_registration_binary'),
    ('IPD sharing rate', 'IPD_sharing_binary'),
    ('DSMB present rate', 'DSMB_binary'),
]

table2b_rows = []
for label, col in indicators:
    pre_n = len(pre_reform)
    pre_s = pre_reform[col].sum()
    post_n = len(post_reform)
    post_s = post_reform[col].sum()
    all_n = len(df_clean)
    all_s = df_clean[col].sum()
    
    pre_p, pre_lo, pre_hi = proportion_ci(int(pre_s), pre_n)
    post_p, post_lo, post_hi = proportion_ci(int(post_s), post_n)
    all_p, all_lo, all_hi = proportion_ci(int(all_s), all_n)
    
    table2b_rows.append({
        'Indicator': label,
        'Pre-Reform': format_proportion(int(pre_s), pre_n, pre_lo, pre_hi),
        'Post-Reform': format_proportion(int(post_s), post_n, post_lo, post_hi),
        'Total': format_proportion(int(all_s), all_n, all_lo, all_hi),
    })

table2b = pd.DataFrame(table2b_rows)
print('Table 2b: Transparency and Safety Oversight Indicators')
print('=' * 80)
print('Format: n/N (% [95% CI Wilson])\n')
table2b

In [ ]:
# ==========================================
# Descriptive statistics for approval-to-registration intervals
# ==========================================

pre_interval = pre_reform['Approval_registration_interval_days'].dropna()
post_interval = post_reform['Approval_registration_interval_days'].dropna()

print('Approval-to-Registration Interval (days) — Descriptive Statistics')
print('=' * 65)
print(f'{"Metric":<30} {"Pre-Reform (n=" + str(len(pre_interval)) + ")":<30} {"Post-Reform (n=" + str(len(post_interval)) + ")":<30}')
print('-' * 65)
print(f'{"Median":<30} {pre_interval.median():<30.0f} {post_interval.median():.0f}')
print(f'{"IQR (Q1–Q3)":<30} {str(int(pre_interval.quantile(0.25))) + "–" + str(int(pre_interval.quantile(0.75))):<30} {str(int(post_interval.quantile(0.25))) + "–" + str(int(post_interval.quantile(0.75)))}')
print(f'{"Mean ± SD":<30} {str(int(pre_interval.mean())) + " ± " + str(int(pre_interval.std())):<30} {str(int(post_interval.mean())) + " ± " + str(int(post_interval.std()))}')
print(f'{"Minimum":<30} {pre_interval.min():<30.0f} {post_interval.min():.0f}')
print(f'{"Maximum":<30} {pre_interval.max():<30.0f} {post_interval.max():.0f}')
print(f'{"Negative values":<30} {int((pre_interval < 0).sum()):<30} {int((post_interval < 0).sum())}')

## 6. Visualization

### Figure 2: Approval-to-registration intervals before and after the reform


In [ ]:
# ==========================================
# Figure 2: swarm plot (beeswarm) + box plot
# ==========================================
# Show approval-to-registration interval distributions before and after the reform

fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [2, 1]})

ax1 = axes[0]  # swarm plot
ax2 = axes[1]  # box plot

# --- Prepare data ---
plot_data = df_clean[['Period', 'Approval_registration_interval_days']].dropna().copy()
plot_data['Period_short'] = plot_data['Period'].map({
    'Pre-Reform (2018-03 to 2023-02)': 'Pre-Reform\n(2018.3–2023.2)',
    'Post-Reform (2023-03 to 2026-07)': 'Post-Reform\n(2023.3–2026.7)',
})

periods = ['Pre-Reform\n(2018.3–2023.2)', 'Post-Reform\n(2023.3–2026.7)']
colors = ['#6BAED6', '#FD8D3C']  # blue / orange

# --- Swarm plot ---
for i, (period, color) in enumerate(zip(periods, colors)):
    subset = plot_data[plot_data['Period_short'] == period]['Approval_registration_interval_days']
    jitter = np.random.normal(0, 0.08, size=len(subset))
    x_pos = i + jitter
    ax1.scatter(x_pos, subset, alpha=0.6, s=60, c=color, edgecolors='white', linewidth=0.5, zorder=2)

# Annotate median + IQR
for i, (period, color) in enumerate(zip(periods, colors)):
    subset = plot_data[plot_data['Period_short'] == period]['Approval_registration_interval_days']
    if len(subset) > 0:
        med = subset.median()
        q1 = subset.quantile(0.25)
        q3 = subset.quantile(0.75)
        
        ax1.plot([i-0.25, i+0.25], [med, med], color='#333333', linewidth=2.5, zorder=3)
        ax1.plot([i-0.25, i+0.25], [q1, q1], color='#333333', linewidth=1.2, linestyle='--', zorder=3)
        ax1.plot([i-0.25, i+0.25], [q3, q3], color='#333333', linewidth=1.2, linestyle='--', zorder=3)
        
        ax1.text(i, 0.98, f'Median={med:.0f}d', transform=ax1.get_xaxis_transform(), ha='center', va='top', fontsize=9, fontweight='bold')

ax1.set_xticks([0, 1])
ax1.set_xticklabels(periods, fontsize=12)
ax1.set_ylabel('Ethics Approval → Registration Interval (days)', fontsize=12)
ax1.set_title('Figure 2A: Distribution of Approval-to-Registration Interval\n(swarm plot)', fontsize=13, fontweight='bold')
ax1.axhline(y=0, color='#CCCCCC', linewidth=0.8, linestyle='-', zorder=1)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

# --- Box plot ---
box_data = [plot_data[plot_data['Period_short'] == p]['Approval_registration_interval_days'].values for p in periods]
bp = ax2.boxplot(box_data, patch_artist=True, widths=0.5,
                  medianprops={'color': '#333333', 'linewidth': 2},
                  whiskerprops={'linewidth': 1.2},
                  capprops={'linewidth': 1.2})

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax2.set_xticklabels(periods, fontsize=12)
ax2.set_ylabel('Ethics Approval → Registration Interval (days)', fontsize=12)
ax2.set_title('Figure 2B: Box Plot', fontsize=13, fontweight='bold')
ax2.axhline(y=0, color='#CCCCCC', linewidth=0.8, linestyle='-')
ax2.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('figure2_approval_registration_interval.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Pre-Reform median: {pre_interval.median():.0f} days (IQR: {pre_interval.quantile(0.25):.0f}–{pre_interval.quantile(0.75):.0f})')
print(f'Post-Reform median: {post_interval.median():.0f} days (IQR: {post_interval.quantile(0.25):.0f}–{post_interval.quantile(0.75):.0f})')

In [ ]:
# ==========================================
# Supplementary Figure: DSMB Presence by Study Phase
# ==========================================
# Explore: do "Other" phase trials (IIT proxy) generally lack DSMB?

fig, ax = plt.subplots(figsize=(10, 6))

# Group by study phase
phase_order = ['I期临床试验', 'I期+II期', '探索性研究/预试验', '其他']
dsmb_by_phase = df_clean.groupby('Study_phase_clean').agg(
    n=('DSMB_binary', 'count'),
    dsmb_yes=('DSMB_binary', 'sum'),
    dsmb_no=('DSMB_binary', lambda x: (x == 0).sum())
).reindex(phase_order).dropna()

# Stacked bar chart
x = np.arange(len(dsmb_by_phase))
width = 0.6

bars_yes = ax.bar(x, dsmb_by_phase['dsmb_yes'], width, label='DSMB Present', color='#6BAED6', edgecolor='white')
bars_no = ax.bar(x, dsmb_by_phase['dsmb_no'], width, bottom=dsmb_by_phase['dsmb_yes'],
                  label='DSMB Absent / Not Determined', color='#FDD0A2', edgecolor='white')

# Annotate counts
for i, (_, row) in enumerate(dsmb_by_phase.iterrows()):
    total = int(row['n'])
    ax.text(i, total + 0.3, f'n={total}', ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
phase_labels_en = {
    'I期临床试验': 'Phase I',
    'I期+II期': 'Phase I+II',
    '探索性研究/预试验': 'Exploratory/Pilot',
    '其他': 'Other',
}
ax.set_xticklabels([phase_labels_en.get(p, p) for p in dsmb_by_phase.index], fontsize=11)
ax.set_ylabel('Number of Trials', fontsize=12)
ax.set_title('DSMB Presence by Study Phase', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('figure_supplement_dsmb_by_phase.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Print proportions
print('DSMB Presence Rate — by Study Phase:')
for phase in phase_order:
    if phase in dsmb_by_phase.index:
        row = dsmb_by_phase.loc[phase]
        pct = row['dsmb_yes'] / row['n'] * 100
        print(f'  {phase}: {int(row["dsmb_yes"])}/{int(row["n"])} ({pct:.1f}%)')
print(f'\n  "Other" (IIT proxy) vs. all other phases combined:')
iit_mask = df_clean['Study_phase_clean'] == '其他'
non_iit_mask = ~iit_mask
iit_dsmb = df_clean.loc[iit_mask, 'DSMB_binary'].sum()
iit_n = iit_mask.sum()
non_iit_dsmb = df_clean.loc[non_iit_mask, 'DSMB_binary'].sum()
non_iit_n = non_iit_mask.sum()
print(f'  IIT proxy ("Other"): {iit_dsmb}/{iit_n} ({iit_dsmb/iit_n*100:.1f}%)')
print(f'  All other phases: {non_iit_dsmb}/{non_iit_n} ({non_iit_dsmb/non_iit_n*100:.1f}%)')

## 7. Statistical testing

> **Protocol note:** When *n* < 30, report descriptive statistics only and do not report *p* values. When *n* ≥ 30 and distributions are non-normal, use the Mann–Whitney U test. Use Fisher's exact test for categorical variables when appropriate.


In [ ]:
# ==========================================
# 7.1 Approval-to-registration interval: Mann–Whitney U test
# ==========================================

n_pre = len(pre_interval)
n_post = len(post_interval)
n_total = n_pre + n_post

print(f'Approval-to-Registration Interval — sample size for testing:')
print(f'  Pre-Reform: {n_pre}')
print(f'  Post-Reform: {n_post}')
print(f'  Total: {n_total}')

if n_total >= 30:
    # Shapiro-Wilk normality test
    if n_pre >= 3:
        shapiro_pre = stats.shapiro(pre_interval)
        print(f'\nShapiro-Wilk normality test (Pre-Reform): W={shapiro_pre.statistic:.3f}, p={shapiro_pre.pvalue:.4f}')
    if n_post >= 3:
        shapiro_post = stats.shapiro(post_interval)
        print(f'Shapiro-Wilk normality test (Post-Reform): W={shapiro_post.statistic:.3f}, p={shapiro_post.pvalue:.4f}')
    
    # Mann-Whitney U (non-parametric, no normality assumption required)
    if n_pre >= 2 and n_post >= 2:
        u_stat, u_p = mannwhitneyu(pre_interval, post_interval, alternative='two-sided')
        print(f'\nMann-Whitney U test:')
        print(f'  U = {u_stat:.0f}')
        print(f'  p = {u_p:.4f}')
        print(f'  Effect size (r = Z/√N): {u_stat / (n_pre * n_post):.3f}')
        
        if u_p < 0.05:
            print(f'  ✅ Significant difference between periods (p < 0.05)')
        else:
            print(f'  ❌ No significant difference between periods (p ≥ 0.05)')
else:
    print(f'\n⚠️ Sample size insufficient (n={n_total} < 30). Per the study protocol, only descriptive statistics are reported; no hypothesis testing is performed.')

In [ ]:
# ==========================================
# 7.2 Categorical variables: Fisher's exact test
# ==========================================

def fisher_test_2x2(pre_success, pre_n, post_success, post_n, label):
    """Fisher's exact test for 2x2 table"""
    pre_fail = pre_n - pre_success
    post_fail = post_n - post_success
    table = [[pre_success, pre_fail], [post_success, post_fail]]
    
    # Check expected frequencies
    total = pre_n + post_n
    expected_00 = (pre_success + post_success) * pre_n / total
    expected_01 = (pre_fail + post_fail) * pre_n / total
    
    odds_ratio, p_value = fisher_exact(table)
    
    print(f'\n{label}:')
    print(f'  Pre-Reform: {pre_success}/{pre_n} ({pre_success/pre_n*100:.1f}%)')
    print(f'  Post-Reform: {post_success}/{post_n} ({post_success/post_n*100:.1f}%)')
    print(f'  Fisher OR = {odds_ratio:.3f}, p = {p_value:.4f}')
    print(f'  Expected frequencies: [{expected_00:.1f}, {expected_01:.1f}] — {"Acceptable" if min(expected_00, expected_01) >= 5 else "⚠️ Expected freq. < 5, interpret with caution"}')
    
    return odds_ratio, p_value

# Pre-registration rate
or_pre, p_pre = fisher_test_2x2(
    int(pre_reform['Prospective_registration_binary'].sum()), len(pre_reform),
    int(post_reform['Prospective_registration_binary'].sum()), len(post_reform),
    'Pre-registration Rate'
)

# IPD sharing rate
or_ipd, p_ipd = fisher_test_2x2(
    int(pre_reform['IPD_sharing_binary'].sum()), len(pre_reform),
    int(post_reform['IPD_sharing_binary'].sum()), len(post_reform),
    'IPD Sharing Rate'
)

# DSMB presence rate
or_dsmb, p_dsmb = fisher_test_2x2(
    int(pre_reform['DSMB_binary'].sum()), len(pre_reform),
    int(post_reform['DSMB_binary'].sum()), len(post_reform),
    'DSMB Presence Rate'
)

## 8. Sensitivity analyses (optional; for the Supplement)


In [ ]:
# ==========================================
# 8.1 Transparency indicators stratified by funding source
# ==========================================

print('Transparency Indicators Stratified by Funding Source')
print('=' * 70)
for fund_cat in ['Industry', 'Academic', 'Self-funded', 'Mixed (Industry+Academic)']:
    subset = df_clean[df_clean['Funding_category'] == fund_cat]
    if len(subset) == 0:
        continue
    print(f'\n{fund_cat} (n={len(subset)}):')
    print(f'  Pre-registration: {subset["Prospective_registration_binary"].sum()}/{len(subset)} ({subset["Prospective_registration_binary"].mean()*100:.1f}%)')
    print(f'  IPD sharing: {subset["IPD_sharing_binary"].sum()}/{len(subset)} ({subset["IPD_sharing_binary"].mean()*100:.1f}%)')
    print(f'  DSMB: {subset["DSMB_binary"].sum()}/{len(subset)} ({subset["DSMB_binary"].mean()*100:.1f}%)')
    if subset['Approval_registration_interval_days'].notna().sum() >= 2:
        interval = subset['Approval_registration_interval_days'].dropna()
        print(f'  Approval→Registration interval median: {interval.median():.0f} (IQR: {interval.quantile(0.25):.0f}–{interval.quantile(0.75):.0f})')

In [ ]:
# ==========================================
# 8.2 Approval-to-registration intervals stratified by study phase
# ==========================================

print('Approval-to-Registration Interval Stratified by Study Phase')
print('=' * 60)
for phase in ['I期临床试验', 'I期+II期', '探索性研究/预试验', '其他']:
    subset = df_clean[df_clean['Study_phase_clean'] == phase]
    if len(subset) == 0:
        continue
    interval = subset['Approval_registration_interval_days'].dropna()
    print(f'{phase} (n={len(subset)}):')
    if len(interval) > 0:
        print(f'  Median: {interval.median():.0f}, IQR: {interval.quantile(0.25):.0f}–{interval.quantile(0.75):.0f}')
    else:
        print(f'  Cannot compute')

In [ ]:
# ==========================================
# 8.3 Transparency stratified by IIT/IND pathway (Supplementary Table S2)
# ==========================================
# This sensitivity analysis tests whether the IIT vs IND regulatory pathway
# is associated with differential transparency and safety oversight.
# Directly supports Discussion 5.3 "dual-track regulatory arbitrage" argument.

print('Supplement Table S2: Transparency Indicators — Stratified by IIT/IND Pathway')
print('=' * 75)

# Exclude records classified as Exclude (basic science)
df_sens = df_clean[df_clean['IIT_classification'] != 'Exclude'].copy()

pathways = ['IIT', 'IND', 'Uncertain']
for pathway in pathways:
    subset = df_sens[df_sens['IIT_classification'] == pathway]
    n = len(subset)
    if n == 0:
        continue
    pre_pct = subset['Prospective_registration_binary'].mean() * 100
    ipd_pct = subset['IPD_sharing_binary'].mean() * 100
    dsmb_pct = subset['DSMB_binary'].mean() * 100
    
    interval = subset['Approval_registration_interval_days'].dropna()
    med_str = f'{interval.median():.0f} (IQR: {interval.quantile(0.25):.0f}–{interval.quantile(0.75):.0f})' if len(interval) >= 2 else 'N/A'
    
    print(f'{pathway:<12} n={n:<4}  Pre-reg: {pre_pct:.1f}%  IPD: {ipd_pct:.1f}%  DSMB: {dsmb_pct:.1f}%  Interval: {med_str}')

print()
print('Note: "Uncertain" records require manual review of ChiCTR sponsor information (Level 3).')
print('These should be resolved before final manuscript submission.')

# --- List Uncertain records for manual review ---
uncertain = df_clean[df_clean['IIT_classification'] == 'Uncertain']
if len(uncertain) > 0:
    print(f'\n{"─" * 80}')
    print(f'RECORDS REQUIRING MANUAL REVIEW (n={len(uncertain)}):')
    print(f'{"─" * 80}')
    print('Check the following on the original ChiCTR page:')
    print('  1. Is the sponsor a pharma/biotech company? → IND')
    print('  2. Does the title mention "IND", "新药", or "注册"? → IND')
    print('  3. Is the responsible unit a hospital/university? → IIT')
    print()
    for _, row in uncertain.iterrows():
        print(f'  [{str(row["注册号"]).strip()}] {str(row["注册题目"])[:80]}')
        print(f'    Funding: {row["Funding_category"]} | Phase: {row["Study_phase_clean"]}')
        print()

## 9. Results export


---

## Appendix: Correspondence with the manuscript Methods section

| Manuscript Methods component | Notebook section |
|---|---|
| Data source: ChiCTR registration data | §2 Data loading |
| Search strategy and eligibility criteria | §3 Eligibility screening |
| Extracted variables: ethics approval date, registration date, registration status, study phase, funding source, IPD sharing, and DSMB | §4 Data cleaning and variable coding |
| Derived measures: approval-to-registration interval and prospective-registration rate | §§4.3 and 4.4a |
| Grouping variable: pre-reform vs. post-reform | §4.2 |
| IIT/IND classification: three-level rule (explicit markers → combined rules → manual review) | §4.7 |
| Statistical methods: median + IQR and Mann–Whitney U test | §6 and §7.1 |
| Proportions + 95% CI and Fisher's exact test | §6 (Table 2b) and §7.2 |
| Figure 2: swarm plot / box plot | §6 |
| Sensitivity analyses by funding source, study phase, and IIT/IND pathway (Supplement) | §8 |

---

## Instructions for use

1. **Before first use:** Confirm that `DATA_PATH` points to the correct `ChiCTR.xlsx` file.
2. **Screening review:** After running §3, carefully check the included and excluded records. Adjust questionable records in the manual-correction code cell.
3. **Funding classification:** `categorize_funding()` uses keyword matching and may misclassify records; manually review the `Funding_category` column.
4. **IIT/IND classification:** `classify_iit()` applies a three-level rule. Records marked `Uncertain` require manual review on the original ChiCTR webpage (see the review list output in §8.3).
5. **Reporting:** When *n* < 30, report only the median and IQR; do not report *p* values.
6. **Data updates:** If `ChiCTR.xlsx` changes, rerun all cells.


---

## Appendix: Correspondence with the manuscript Methods section

| Manuscript Methods component | Notebook section |
|---|---|
| Data source: ChiCTR registration data | §2 Data loading |
| Search strategy and eligibility criteria | §3 Eligibility screening |
| Extracted variables: ethics approval date, registration date, registration status, study phase, funding source, IPD sharing, and DSMB | §4 Data cleaning and variable coding |
| Derived measures: approval-to-registration interval and prospective-registration rate | §§4.3 and 4.4a |
| Grouping variable: pre-reform vs. post-reform | §4.2 |
| Statistical methods: median + IQR and Mann–Whitney U test | §5 and §7.1 |
| Proportions + 95% CI and Fisher's exact test | §5 (Table 2b) and §7.2 |
| Figure 2: swarm plot / box plot | §6 |
| Sensitivity analyses (Supplement) | §8 |

---

## Instructions for use

1. **Before first use:** Confirm that `DATA_PATH` points to the correct `ChiCTR.xlsx` file.
2. **Screening review:** After running §3, carefully check the included and excluded records. Adjust questionable records in the manual-correction code cell.
3. **Funding classification:** `categorize_funding()` uses keyword matching and may misclassify records; manually review the `Funding_category` column.
4. **Reporting:** When *n* < 30, report only the median and IQR; do not report *p* values.
5. **Data updates:** If `ChiCTR.xlsx` changes, rerun all cells.
